# Diagnose an empty IPTW run

Companion to [05b_run_biomarker_pipeline.ipynb](05b_run_biomarker_pipeline.ipynb), for when stage 5
finishes without error but the CSVs in `biomarker_analysis/IPTW_runs_*/` have no rows.

### How a run finishes clean and still produces nothing

Every marker fit starts with `filter_finite_rows(df.select(cols), cols)`, where `cols` is the
outcome columns plus `base_vars` plus the marker. `filter_finite_rows` casts each column to `Float64`
with `strict=False`, so a single **non-numeric** covariate becomes all-null, fails `is_finite()`, and
drops every row of the model frame. `CoxPHFitter` then raises for every marker in turn, `_safe_fit`
catches it, `results` comes back empty, and the screen writes a zero-row parquet. `compile_IPTW_results`
reads that without complaint and reports "0 significant hits" — so a broken run and a genuine null
result look identical on disk.

That is what the raw `CANCER_TYPE` label did: `build_cancer_type_df` deliberately keeps the string
column beside its dummies, and the pan-cancer branch selected covariates with a loose
`'CANCER_TYPE' in c` substring test, which swept it into `base_vars`. The covariate filters are now
anchored on the dummy prefixes (`CANCER_TYPE_`, `PANEL_VERSION_`), a numeric guard runs before each
screen, and a screen where every fit fails raises instead of writing an empty file.

### What this notebook does

1. **Guard tests** — the invariants that keep a silent empty run from recurring.
2. **Result inventory** — row counts, not byte sizes, for everything the last run wrote. A gzipped
   zero-row parquet still carries a schema and a footer, so `ls -l` looks perfectly plausible.
3. **Input diagnosis** — rebuilds `base_vars` from the saved `IPTW_df_*.parquet` exactly as
   `run_IPTW_analysis.main()` does, names any non-numeric covariate, attributes row loss per column,
   counts markers with support, and runs one trial fit with the traceback `_safe_fit` would have hidden.

Read-only throughout — nothing here writes to `DATA_PATH`.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path


def find_v2_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find v2 root from {start}")


V2_ROOT = find_v2_root()
REPO_ROOT = V2_ROOT.parent
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

import config
import polars as pl

BIOMARKER_PATH = config.BIOMARKER_PATH

# Mirrors the grids in run_IPTW_analysis; change them in the script, not here.
COHORTS = ["cohort1", "cohort2"]
PS_MODELS = ["covariates_only", "covariates_plus_embeddings"]
SPECS = [f"{c}_{p}" for c in COHORTS for p in PS_MODELS]

print(f"v2 root:   {V2_ROOT}")
print(f"repo root: {REPO_ROOT}")
print(f"Python:    {sys.executable}")
print(f"Biomarker: {BIOMARKER_PATH}")

## 1. Guard tests

`tests/test_iptw_guards.py` pins the three invariants that make a silent empty run impossible:
the raw `CANCER_TYPE`/`PANEL_VERSION` string labels are rejected as model covariates, the prefix
filters that build `base_vars` exclude them, and `_run_marker_screen` raises when every fit fails
(while still tolerating partial failure and an empty marker list).

These `importorskip` on `statsmodels` and `zstandard`, so they **skip** in environments without
them — a "skipped" line here means the test never ran, not that it passed.

In [ ]:
proc = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "tests/test_iptw_guards.py"],
    cwd=REPO_ROOT,
)
if proc.returncode == 0:
    print("\nGuards hold.")
elif proc.returncode == 5:
    # pytest's NO_TESTS_COLLECTED: every test skipped, so nothing actually ran.
    print("\n[skipped] No tests collected — statsmodels/zstandard are missing from this "
          "environment, so the guards were never exercised. Run this on the cluster kernel.")
else:
    print(f"\n[FAIL] pytest exited {proc.returncode} — the guards are not in place; "
          "fix that before trusting anything below.")

## 2. What the last run wrote

Row counts for every result parquet already on disk. `EMPTY (no rows)` is the signature of a screen
whose fits all failed — the file exists, is valid gzip, has the right columns, and carries no data.

The per-cancer-type screens build `base_vars` without the cancer-type dummies, so if only the
`pan_cancer_*` rows are empty, the raw-label bug explains it completely. Empty per-type files mean
something else as well, and section 3 will say what.

In [ ]:
def result_rows(path: str) -> int | None:
    """Rows in a result parquet, from its metadata, or None if it cannot be read."""
    try:
        return pl.scan_parquet(path).select(pl.len()).collect().item()
    except (OSError, pl.exceptions.PolarsError):
        return None


total_files = 0
empty_files = 0

for spec in SPECS:
    run_path = os.path.join(BIOMARKER_PATH, f"IPTW_runs_{spec}/")
    print(f"\n{'=' * 78}\n{spec}\n{'=' * 78}")
    if not os.path.isdir(run_path):
        print("  no directory — stage 5 has not run for this spec")
        continue

    found = False
    for name in sorted(f for f in os.listdir(run_path) if f.endswith(".parquet")):
        found = True
        total_files += 1
        n_rows = result_rows(os.path.join(run_path, name))
        if n_rows is None:
            print(f"  {name:<58} UNREADABLE")
        elif n_rows == 0:
            empty_files += 1
            print(f"  {name:<58} {0:>6} rows   <- EMPTY (no rows)")
        else:
            print(f"  {name:<58} {n_rows:>6} rows")
    if not found:
        print("  directory exists but holds no .parquet — stage 5 stopped before writing")

print(f"\n{'=' * 78}")
if total_files == 0:
    print("No result files anywhere — stage 5 has not produced output yet. "
          "Section 3 still checks whether its inputs are sound.")
else:
    print(f"{empty_files}/{total_files} result files have no rows.")
if total_files and empty_files == total_files:
    print("Every screen failed. Section 3 names the covariate responsible.")
elif empty_files:
    print("Some screens produced results. Compare which cancer types are empty against "
          "the base_vars each branch builds.")

## 3. Input diagnosis

`pipelines.biomarkers.diagnose_iptw_inputs` reads the saved `IPTW_df_*.parquet` — no refit, no rerun,
seconds per spec — and for each specification reports:

- patient count, ICI/control split, and death count
- every non-numeric column in the frame
- the `pan_cancer` `base_vars` it would assemble, flagging any non-numeric member
- rows surviving `filter_finite_rows` on `base_vars` alone, **before** a marker is added — `0/N` here
  is the bug, stated outright
- rows lost per covariate in isolation, so the culprit is named even when several columns are
  numeric-but-null
- markers passing within-arm support
- one real fit per track with `_safe_fit` removed, so the traceback is visible

Run it as a subprocess, matching 05b, so nothing leaks into this kernel.

Set `MAX_MARKERS` below for a fast pass: it caps the per-marker support scans, which are the
slow part (one numpy pass per marker, ~1500 markers x 2 tracks x 4 specs). Everything
structural — schema, rank, empty model frame, the trial fits — is marker-independent and still
runs in full. It is the same `IPTW_MAX_MARKERS` the analysis script uses, seeded identically,
so a capped diagnosis inspects exactly the markers a capped run screened.

In [ ]:
# Cap the per-marker support scans — the slow part of this script (a numpy pass
# per marker, ~1500 markers x 2 tracks x 4 specs). The structural checks (schema,
# rank, empty model frame) and the trial fits do not depend on the marker list,
# so a cap gives the same diagnosis in a fraction of the time.
#
# This is the same env var run_IPTW_analysis uses, and both scripts seed the RNG
# identically, so a cap here inspects exactly the markers a smoke run screened.
# None => scan every marker.
MAX_MARKERS: int | None = None   # e.g. 50 for a fast pass

DIAG_ENV = os.environ.copy()
if MAX_MARKERS is not None:
    DIAG_ENV["IPTW_MAX_MARKERS"] = str(MAX_MARKERS)
    print(f"Capping support scans at {MAX_MARKERS} markers per spec.\n")
else:
    # A stale value in the kernel's environment would silently cap this run and
    # make the support counts look wrong for reasons invisible in the notebook.
    DIAG_ENV.pop("IPTW_MAX_MARKERS", None)
    DIAG_ENV.pop("IPTW_MARKER_FRACTION", None)

proc = subprocess.run(
    [sys.executable, "-m", "pipelines.biomarkers.diagnose_iptw_inputs"],
    cwd=V2_ROOT, env=DIAG_ENV,
)
print(f"\n[exit {proc.returncode}]")

## Reading the output

| What section 3 shows | What it means | What to do |
|---|---|---|
| `NON-NUMERIC base_vars` naming a column | That column empties every model frame | Exclude it from the covariate filter in `run_IPTW_analysis.main()`, or dummy-code it in `generate_IPTW_df` |
| `Rows surviving ... 0/N` | Confirmed: no fit could have succeeded | As above — then re-run stage 5 |
| Trial fit raises with something else | A real modelling failure, not a plumbing one | Read the traceback; `_safe_fit` would have hidden it |
| `IPTW_df: 0 patients` | The break is upstream | Re-run stage 4 (`generate_IPTW_df`) and check its join funnel |
| `Markers with within-arm support: 0/N` | Cohort too small or too imbalanced for the support thresholds | Check `MIN_MARKER_POS_PER_ARM` / `MIN_EVENTS_PER_MARKER_GROUP` against the cohort sizes printed above |
| Everything numeric, markers have support, trial fits OK | The inputs are sound now | Re-run stage 5 in 05b (`RUN_COX = True`, `SKIP_IF_DONE = False`) |

After a re-run, stage 5 can no longer end this way quietly: `_run_marker_screen` raises with a tally
of the distinct fit errors rather than writing an empty file, and
`assert_model_covariates_numeric` fails before any screen starts.
